# 購買バスケット分析: アプリオリ法による関連ルール抽出

このノートブックでは、レストランの注文データを使って、商品カテゴリの同時購買パターンを分析します。
GitHub で読みやすいように、手順を最小限に整理し、再利用しやすい関数構成にまとめています。

## 前提
- 必要パッケージ: pandas, mlxtend, openpyxl
- 入力ファイル: `ds_indian_restaurants_orders_daypart_foodtype.csv`
- 出力先: `association_rules_output` フォルダー

In [6]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

DATA_PATH = Path("ds_indian_restaurants_orders_daypart_foodtype.csv")
OUTPUT_DIR = Path("association_rules_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# 最初の探索としては 0.07 前後が扱いやすい設定です。
# 低すぎると候補集合が急増してノイズも増えやすく、高すぎると有用な組み合わせを落としやすくなります。
# 取引件数が少ない場合や商品カテゴリが細かい場合は 0.03 から 0.05 に下げ、
# ルールが多すぎる場合は 0.10 以上へ上げる、という順で調整するのが基本です。
MIN_SUPPORT = 0.07

# lift は「偶然よりどれだけ強く同時購買されるか」を見る指標です。
# まずは 1.5 を基準にして、弱い関係まで拾いたい場合は 1.2 前後へ下げ、
# 強い関係だけ残したい場合は 2.0 以上へ上げて比較します。
MIN_LIFT = 1.5

# confidence は「前件が起きたときに後件が続く割合」です。
# 実務では 0.6 から 0.8 を比較し、施策に使える安定性があるかどうかで判断します。
MIN_CONFIDENCE = 0.70

# 表示件数は確認用です。
# 候補比較を丁寧に行いたい場合は 30 から 50 に増やします。
TOP_N = 20

In [2]:
orders = pd.read_csv(DATA_PATH)

required_columns = {"Order Number", "Item Name", "brand_tag", "day_part", "food_type"}
missing_columns = required_columns.difference(orders.columns)
if missing_columns:
    raise ValueError(f"必要な列が見つかりません: {sorted(missing_columns)}")

# day_part はデータ上で「夕食」「昼食」と記録されているため、この表記をそのまま使います。
display(orders.head())
display(
    orders.groupby(["brand_tag", "day_part"]).size().rename("rows").reset_index()
)

,Order Number,Order Date,Item Name,Quantity,Total products,brand_tag,day_part,food_type
0,16118,03/08/2019 20:25,Plain Papadum,2,6,res_1,夕食,Papadum
1,16118,03/08/2019 20:25,King Prawn Balti,1,6,res_1,夕食,Prawn Balti
2,16118,03/08/2019 20:25,Garlic Naan,1,6,res_1,夕食,Naan
3,16118,03/08/2019 20:25,Mushroom Rice,1,6,res_1,夕食,Rice
4,16118,03/08/2019 20:25,Paneer Tikka Masala,1,6,res_1,夕食,Tikka Masala


,brand_tag,day_part,rows
0,res_1,夕食,70408
1,res_1,昼食,4410
2,res_2,夕食,114898
3,res_2,昼食,4285


## 関数定義

重複した前処理とルール抽出を関数化し、店舗や時間帯を増やす場合でも同じ流れを再利用できるようにします。

In [3]:
def build_basket_table(data, brand_tag, day_part):
    """指定した店舗と時間帯の取引を、アプリオリ法に入力しやすい真偽値表へ変換します。"""
    filtered = data.loc[
        (data["brand_tag"] == brand_tag) & (data["day_part"] == day_part)
    ]

    if filtered.empty:
        raise ValueError(
            f"条件に一致するデータがありません: brand_tag={brand_tag}, day_part={day_part}"
        )

    basket = (
        filtered.groupby(["Order Number", "food_type"])["Item Name"]
        .count()
        .unstack(fill_value=0)
        .sort_index(axis=1)
    )

    # apriori は 0/1 も受け取れますが、真偽値にしておくと意図が明確です。
    return basket.gt(0)


def mine_rules(basket, min_support=MIN_SUPPORT, metric="lift", min_threshold=1.0):
    """頻出項目集合と関連ルールを作成して返します。"""
    frequent_itemsets = apriori(basket, min_support=min_support, use_colnames=True)

    if frequent_itemsets.empty:
        return frequent_itemsets, pd.DataFrame()

    rules = association_rules(frequent_itemsets, metric=metric, min_threshold=min_threshold)

    if rules.empty:
        return frequent_itemsets, rules

    return frequent_itemsets, rules.sort_values(
        by=["lift", "confidence", "support"],
        ascending=False,
    ).reset_index(drop=True)


def select_rules(rules, min_lift=MIN_LIFT, min_confidence=MIN_CONFIDENCE):
    """閲覧しやすいように lift と confidence の下限で絞り込みます。"""
    if rules.empty:
        return rules

    selected = rules.loc[
        (rules["lift"] >= min_lift) & (rules["confidence"] >= min_confidence)
    ].copy()

    return selected.sort_values(
        by=["lift", "confidence", "support"],
        ascending=False,
    ).reset_index(drop=True)


def stringify_itemset(value):
    """frozenset を読みやすい文字列へ変換します。"""
    return ", ".join(sorted(value))


def prepare_export_table(rules):
    """Excel 出力前に集合表現を文字列へ変換します。"""
    if rules.empty:
        return rules

    export_table = rules.copy()
    export_table["antecedents"] = export_table["antecedents"].map(stringify_itemset)
    export_table["consequents"] = export_table["consequents"].map(stringify_itemset)
    return export_table

## 分析実行

4 つの条件を同じ設定で比較します。最初は共通しきい値で全体像をつかみ、必要に応じて店舗ごとに `MIN_SUPPORT` や `MIN_CONFIDENCE` を微調整するのが扱いやすい進め方です。

In [4]:
analysis_targets = [
    {"brand_tag": "res_1", "day_part": "夕食", "label": "店舗1_夕食"},
    {"brand_tag": "res_2", "day_part": "夕食", "label": "店舗2_夕食"},
    {"brand_tag": "res_1", "day_part": "昼食", "label": "店舗1_昼食"},
    {"brand_tag": "res_2", "day_part": "昼食", "label": "店舗2_昼食"},
]

analysis_results = {}
summary_rows = []

for target in analysis_targets:
    label = target["label"]
    basket = build_basket_table(orders, target["brand_tag"], target["day_part"])
    frequent_itemsets, rules = mine_rules(basket)
    selected_rules = select_rules(rules)

    export_all_rules = prepare_export_table(rules)
    export_selected_rules = prepare_export_table(selected_rules)

    all_rules_path = OUTPUT_DIR / f"{label}_all_rules.xlsx"
    selected_rules_path = OUTPUT_DIR / f"{label}_selected_rules.xlsx"

    if not export_all_rules.empty:
        export_all_rules.to_excel(all_rules_path, index=False)
    if not export_selected_rules.empty:
        export_selected_rules.to_excel(selected_rules_path, index=False)

    analysis_results[label] = {
        "basket": basket,
        "frequent_itemsets": frequent_itemsets,
        "rules": rules,
        "selected_rules": selected_rules,
        "export_selected_rules": export_selected_rules,
        "all_rules_path": all_rules_path,
        "selected_rules_path": selected_rules_path,
    }

    summary_rows.append(
        {
            "対象": label,
            "取引数": len(basket),
            "商品カテゴリ数": basket.shape[1],
            "頻出項目集合数": len(frequent_itemsets),
            "全ルール数": len(rules),
            "採用ルール数": len(selected_rules),
        }
    )

    print(f"=== {label} ===")
    if export_selected_rules.empty:
        print("条件を満たすルールは見つかりませんでした。しきい値を緩めるか、カテゴリ粒度を見直してください。")
        print()
    else:
        display(
            export_selected_rules[
                ["antecedents", "consequents", "support", "confidence", "lift"]
            ].head(TOP_N)
        )
        print(f"保存先: {selected_rules_path}")
        print()

=== 店舗1_夕食 ===


,antecedents,consequents,support,confidence,lift
0,"Chutney, Sauce",Papadum,0.079397,0.854185,2.905300
1,"Chutney, Rice",Papadum,0.131606,0.808374,2.749487
2,"Chutney, Naan, Rice",Papadum,0.087738,0.795058,2.704195
3,Chutney,Papadum,0.150614,0.767784,2.611429
4,"Chutney, Naan",Papadum,0.098404,0.763060,2.595360
5,"Rice, Sauce",Papadum,0.103296,0.719955,2.448751


保存先: association_rules_output\店舗1_夕食_selected_rules.xlsx

=== 店舗2_夕食 ===


,antecedents,consequents,support,confidence,lift
0,"Chutney, Sauce","Papadum, Rice",0.081996,0.747328,2.744249
1,"Chutney, Rice, Sauce",Papadum,0.081996,0.920407,2.660924
2,"Chutney, Naan, Sauce",Papadum,0.075119,0.916125,2.648544
3,"Chutney, Sauce",Papadum,0.100069,0.912051,2.636765
4,"Chutney, Naan","Papadum, Rice",0.113398,0.717611,2.635128
5,"Bhaji, Chutney",Papadum,0.074799,0.898207,2.596744
6,"Aloo, Chutney",Papadum,0.073466,0.890756,2.575203
7,"Chutney, Naan, Rice",Papadum,0.113398,0.878926,2.541000
8,"Chutney, Naan",Papadum,0.137655,0.871120,2.518434
9,"Chutney, Rice",Papadum,0.149757,0.867779,2.508774


保存先: association_rules_output\店舗2_夕食_selected_rules.xlsx

=== 店舗1_昼食 ===


,antecedents,consequents,support,confidence,lift
0,"Aloo, Chicken",Rice,0.078664,0.924051,2.046585
1,Aloo,Rice,0.135776,0.845638,1.872916
2,Chicken,Rice,0.149784,0.789773,1.749186
3,"Chicken, Naan",Rice,0.073276,0.772727,1.711434


保存先: association_rules_output\店舗1_昼食_selected_rules.xlsx

=== 店舗2_昼食 ===


,antecedents,consequents,support,confidence,lift
0,"Naan, Papadum",Rice,0.072142,0.866667,2.352008
1,"Chicken, Naan",Rice,0.099889,0.789474,2.142517
2,"Chicken, Rice",Naan,0.099889,0.750000,2.085648
3,Tikka Masala,Rice,0.084351,0.737864,2.002456
4,"Papadum, Rice",Naan,0.072142,0.706522,1.964741


保存先: association_rules_output\店舗2_昼食_selected_rules.xlsx



In [5]:
summary_df = pd.DataFrame(summary_rows).sort_values("対象").reset_index(drop=True)
display(summary_df)

,対象,取引数,商品カテゴリ数,頻出項目集合数,全ルール数,採用ルール数
0,店舗1_夕食,12469,108,55,152,6
1,店舗1_昼食,928,107,27,32,4
2,店舗2_夕食,18757,120,77,294,15
3,店舗2_昼食,901,113,28,36,5


## 調整の目安

- 採用ルール数が少なすぎる場合は、`MIN_SUPPORT` を 0.01 から 0.02 刻みで下げて変化を見ます。
- 候補が多すぎる場合は、まず `MIN_SUPPORT` を上げ、その次に `MIN_CONFIDENCE` を上げて絞り込みます。
- 強い組み合わせだけを見たい場合は、`MIN_LIFT` を上げます。
- 店舗ごとの取引件数に差があるため、最終的には全店舗共通値ではなく、店舗別にしきい値を持たせると解釈しやすくなります。
